# EXP-022 EL8 final-model handoff

**E50+L remains production. These notebooks do not score Development, open locked labels or upload to Codabench.**

Upload the private `el8_refit_handoff.zip` once (Drive for Colab; private Dataset for Kaggle). Select a CUDA GPU, preferably the original **L4**, and enable Internet for the pinned Python/dependencies. Set the input path below and Run all. Preparation validates the recovered 60 CV models and prints the cost forecast. Training is **off by default**.

After approving the final-fit policy and GPU budget, set `AUTHORIZE_FINAL_REFIT=True` and a positive `MAX_GPU_HOURS` (for example 1.5 hours), then Run all once. This automatically runs strict CV replay, memory/parity preflight, timing pilot, the three missing final seeds, checkpoint replay, image-only smoke and full export. A parity failure stops before final training. **Cross-runtime parity has failed on the local RTX 4060; L4 replay of this deployment implementation remains unverified.**

The new final policy uses original EXP-022 loss, learning rates, LoRA, seeds and nested stopping/refit; it specifies a newly reviewed fold=0 hash for the full-population inner split. No CV model is treated as a final model.

Logs, completed seeds and export receipts persist under `PERSISTENT_ROOT`. On interruption Run all again with the same paths and runtime; completed seeds are checked and reused, while an interrupted current seed restarts. Colab Drive survives runtime deletion. **Kaggle working files require Save Version/output download before session loss; a completed export alone does not guarantee persistence outside the Kaggle session.** Final export ZIP + `.sha256` supports offline import without the old kernel. Setup may take several minutes; no dependency versions are substituted.


In [ ]:
PLATFORM = 'Kaggle'
HANDOFF_ZIP = '/kaggle/input/your-private-dataset/el8_refit_handoff.zip'
HANDOFF_SHA256 = '7cc9a836e4d5eb3db90252d4c2e3bf9652667d5e46c8c47b65398c09d1f7386b'
IMPLEMENTATION_REVISION = 'a6fd6559e0b08bde78f12100183dab9ff8ce2fe4'
HELPER_SHA256 = '4575e0ba93735d8a21fef69abc0bdb241beba0bc52eed6628e6d275c4e0a854c'
LOCAL_ROOT = '/kaggle/temp/el8_final'
PERSISTENT_ROOT = '/kaggle/working/el8_final_runs'
# Approval includes configs/el8_final_refit.json: fold=0 hashed 10% stopping split,
# original 100 epoch maximum / patience 10 / fresh full-1335-painting refit.
AUTHORIZE_FINAL_REFIT = False
MAX_GPU_HOURS = 0.0


In [ ]:
import hashlib, importlib.util, json, subprocess, zipfile
from pathlib import Path

if PLATFORM == 'Colab':
    from google.colab import drive
    drive.mount('/content/drive')

archive = Path(HANDOFF_ZIP)
if not archive.is_file():
    raise FileNotFoundError(f'Upload the private el8_refit_handoff.zip and check HANDOFF_ZIP: {archive}')
digest = hashlib.sha256()
with archive.open('rb') as handle:
    for chunk in iter(lambda:handle.read(1024*1024), b''):digest.update(chunk)
if digest.hexdigest() != HANDOFF_SHA256:
    raise RuntimeError('Handoff ZIP hash mismatch. Use the producer receipt; never replace the expected hash.')
local = Path(LOCAL_ROOT);local.mkdir(parents=True, exist_ok=True)
bundle = local/'bootstrap-source.bundle'
with zipfile.ZipFile(archive) as z:
    manifest = json.loads(z.read('artifact_manifest.json'))
    if manifest['metadata']['deployment_revision'] != IMPLEMENTATION_REVISION:
        raise RuntimeError('Handoff deployment revision differs from the notebook pin.')
    raw = z.read('source/source.bundle')
    if hashlib.sha256(raw).hexdigest() != manifest['files_sha256']['source/source.bundle']:
        raise RuntimeError('Source bundle hash mismatch.')
    if bundle.exists() and bundle.read_bytes() != raw:
        raise RuntimeError('Existing source bundle differs; preserve it and choose a fresh LOCAL_ROOT.')
    bundle.write_bytes(raw)
repo = local/'deployment'
if not repo.exists():
    subprocess.run(['git','clone','--no-checkout',str(bundle),str(repo)],check=True)
    subprocess.run(['git','-C',str(repo),'checkout','--detach',IMPLEMENTATION_REVISION],check=True)
if subprocess.check_output(['git','-C',str(repo),'rev-parse','HEAD'],text=True).strip() != IMPLEMENTATION_REVISION:
    raise RuntimeError('Source checkout differs; choose a fresh LOCAL_ROOT.')
helper = repo/'scripts/el8_cloud.py'
if hashlib.sha256(helper.read_bytes()).hexdigest() != HELPER_SHA256:
    raise RuntimeError('Cloud launcher hash mismatch; keep the published pins.')
spec = importlib.util.spec_from_file_location('el8_cloud_launch',helper)
workflow = importlib.util.module_from_spec(spec);spec.loader.exec_module(workflow)
RESULT = workflow.launch(archive,HANDOFF_SHA256,LOCAL_ROOT,PERSISTENT_ROOT,
                        platform=PLATFORM,authorize=AUTHORIZE_FINAL_REFIT,max_gpu_hours=MAX_GPU_HOURS)
print(json.dumps(RESULT,indent=2))
